In [ ]:
from transformers import AutoTokenizer

# Load tokenizer of model that should be fine-tuned
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")

In [ ]:
from collections import defaultdict
from src.data_loaders.spider_ent_data import get_spider_ent_data
from src.data_loaders.spider_data import get_spider_train, get_spider_val
overflow_spider_train = defaultdict(list)   
overflow_spider_val = defaultdict(list)   
overflow_spider_ent = defaultdict(list)   

spider_train_data = get_spider_train(tokenizer, 3000, overflow_spider_train)
spider_val_data = get_spider_val(tokenizer, 3000, overflow_spider_val)
                   # db_id -> Tokens drüber pro Split
data = get_spider_ent_data(tokenizer, 3000, overflow_spider_ent)

In [ ]:
def print_overflow_stats(overflow):
    import pandas as pd
    
    df = pd.DataFrame(
        [(k, x) for k, vs in overflow.items() for x in vs],
        columns=["db", "value"],
    )
    stats = df.groupby("db")["value"].agg(["count", "mean", "median", "std", "min", "max"])
    print(stats)

print("Spider-Train: ")
print_overflow_stats(overflow_spider_train)
print("Spider-Val: ")
print_overflow_stats(overflow_spider_val)
print("Spider-Ent: ")
print_overflow_stats(overflow_spider_ent)

In [ ]:
from src.data_loaders.spider_ent_data import get_spider_ent_data
spider_ent_data = get_spider_ent_data(tokenizer)

In [ ]:
spider_ent_data
print()

In [ ]:
from src.data_loaders.spider_data import get_spider_train, get_spider_val

spider_train_data = get_spider_train()
spider_val_data = get_spider_val()
print()

In [ ]:
import re
from transformers import AutoTokenizer
from src.data_loaders.spider_data import get_spider_val

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
data = get_spider_val(tok, 3000)

miss, total = 0, 0
for item in data:
    cand = set()
    for prompt in item["input"]:
        # Regex-Lücken sichtbar machen:
        n_marker = prompt.count("«")
        found = re.findall(r"«\s+(\S+)\s+(\S+)\s*»", prompt)
        if len(found) != n_marker:
            print(f"Regex verliert {n_marker - len(found)} Kandidaten")
        cand |= {(t.lower(), c.lower()) for t, c in found}
    gold = {(t.lower(), c.lower()) for t, cols in item["gold_schema"].items() for c in (cols or [])}
    missing = gold - cand
    miss += len(missing); total += len(gold)
    if missing and miss < 30:
        print("Fehlt:", missing, "| Kandidaten-Beispiel:", sorted(cand)[:3])

print(f"\nStrukturell unerreichbar: {miss}/{total} = {miss/total:.2%}")

In [ ]:
pip install dotenv

In [ ]:
import json
dev = json.load(open("data/spider/dev.json"))  # Pfad ggf. anpassen
bad = [ex for ex in dev if ex["db_id"] == "concert_singer" and " AS T1" in ex["query"].upper()]
ex = bad[0]
print(ex["question"])
print(ex["query"])

In [ ]:
from src.data_loaders.spider_data import get_spider_train

data = get_spider_train(10000)
for item in data:
    for columns in item["gold_schema"].values():
        if len(columns) == 0:
            print("No columns", item['gold_schema'], item['sql'])

In [ ]:
from src.QLora.train import build_training_samples

raw_data = get_spider_train(10000)
samples = build_training_samples(raw_data)
print()

In [ ]:
import re
from src.data_loaders.spider_data import get_spider_schema_ddl_and_candidates, spider_train
from src.utils import group_tables_by_fk_component, create_schema_linker_input, _extract_fk_targets

DB_ID = "baseball_1"
MAX_TOKENS = 3000  # ggf. anpassen, z.B. 1024 wie in train.py

_TABLE_NAME_IN_CHUNK = re.compile(r'CREATE TABLE\s+`?(\S+?)`?\s*\(')

def tables_in_chunk(chunk: str) -> list[str]:
    return [name.lower() for name in _TABLE_NAME_IN_CHUNK.findall(chunk)]

db_tables = get_spider_schema_ddl_and_candidates()[DB_ID]
name_to_table = {t["candidates"]["table"].lower(): t for t in db_tables}

# --- 1) FK-Gruppen ---
groups = group_tables_by_fk_component(db_tables)
print(f"{DB_ID}: {len(db_tables)} Tabellen, {len(groups)} FK-Gruppe(n)\n")
for i, group in enumerate(groups, start=1):
    names = [t["candidates"]["table"] for t in group]
    print(f"Gruppe {i} ({len(names)} Tabellen): {names}")

# --- 2) Tatsaechliches Chunking mit dem geladenen Tokenizer ---
question_text = next(q["question"] for q in spider_train if q["db_id"] == DB_ID)
print(f"\nBeispiel-Frage: {question_text!r}")

chunks = create_schema_linker_input(db_tables, question_text, MAX_TOKENS, DB_ID, tokenizer)
print(f"\n=> {len(chunks)} Chunk(s) bei max_tokens={MAX_TOKENS}\n")

chunk_tables = [tables_in_chunk(c) for c in chunks]
for i, names in enumerate(chunk_tables, start=1):
    print(f"Chunk {i} ({len(names)} Tabellen): {names}")

# --- 3) FK-Referenzen, die in einem ANDEREN Chunk landen ---
print("\nFehlende FK-Referenzen pro Chunk:")
found_any = False
for i, names in enumerate(chunk_tables, start=1):
    for name in names:
        table = name_to_table.get(name)
        if not table:
            continue
        targets = _extract_fk_targets(table.get("ddl") or "")
        missing = [t for t in targets if t in name_to_table and t not in names]
        if missing:
            found_any = True
            print(f"  Chunk {i}: '{name}' referenziert {missing}, die NICHT in Chunk {i} sind")
if not found_any:
    print("  (keine - alle FK-Referenzen sind innerhalb desselben Chunks)")
